# Tugas 1 — Milestone 3: Analisis Temporal, Spasial, dan Advanced EDA
**Mata Kuliah:** Analisis Big Data

**Target Check-in:** Pertemuan 8 (Mulai dikerjakan Pertemuan 6)

**Nama Mahasiswa:** [Imam Prayuda]

**NIM:** [202310370311114]

**Username GitHub:** [IMAMPRAYUDA114]

## Panduan Pengerjaan Milestone 3:

1. **Sumber Data:** Gunakan dataset yang telah dibersihkan pada Milestone 2 (`02_data_cleaning.ipynb`).
2. **Analisis Temporal / Windowing:** Terapkan minimal satu operasi temporal (Polars `group_by_dynamic()`, `rolling_mean()`, `shift()`, atau DuckDB window function `OVER (...)`).
3. **Visualisasi Interaktif:** Bangun minimal 6 visualisasi interaktif informatif menggunakan Plotly Express / Graph Objects atau Altair.
4. **Perumusan Insight:** Tuliskan minimal 5 temuan atau wawasan analitik mendalam berdasarkan hasil visualisasi data di atas.

## Bagian 1: Inisialisasi dan Memuat Data Bersih
Muat dataset hasil pembersihan dari folder `data/` menggunakan Polars.

In [2]:
from pathlib import Path

import duckdb
import polars as pl
import plotly.graph_objects as go

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"
train_path = DATA_DIR / "commonlingua_train_clean.parquet"
validation_path = DATA_DIR / "commonlingua_validation_clean.parquet"

missing_files = [
    path for path in (train_path, validation_path)
    if not path.exists()
]
if missing_files:
    missing_list = "\n".join(str(path) for path in missing_files)
    raise FileNotFoundError(
        f"File hasil cleaning tidak ditemukan:\n{missing_list}\n"
        "Jalankan notebooks/02_data_cleaning.ipynb terlebih dahulu."
    )

df_train = pl.read_parquet(train_path).with_columns(
    pl.lit("train").alias("dataset_split")
)
df_validation = pl.read_parquet(validation_path).with_columns(
    pl.lit("validation").alias("dataset_split")
)

df = pl.concat([df_train, df_validation], how="diagonal_relaxed")

print(f"Data loaded: {df.height:,} baris x {df.width} kolom")
print("Kolom:", df.columns)

Data loaded: 2,121,704 baris x 11 kolom
Kolom: ['text', 'lang', 'source', 'identifier', 'title', 'collection', 'license', 'open_type', 'creator', 'date', 'dataset_split']


## Bagian 2: Analisis Temporal dan Window Functions (Materi Pertemuan 6)
Terapkan teknik windowing Polars (`group_by_dynamic`, `rolling_mean`, `shift`, atau DuckDB window functions) untuk menangkap tren waktu, pola musiman, atau pergerakan akumulatif.

In [3]:
if "date" not in df.columns:
    raise ValueError("Kolom 'date' tidak ditemukan pada dataset.")

date_text = pl.col("date").cast(pl.String).str.strip_chars()

year_text = (
    pl.when(date_text.str.contains(r"^-?\d{1,4}$"))
    .then(date_text)
    .when(date_text.str.contains(r"^\d{4}[-/]\d{1,2}"))
    .then(date_text.str.extract(r"^(\d{4})", 1))
    .otherwise(None)
)

year_counts = (
    df.lazy()
    .with_columns(year_text.alias("_year_text"))
    .with_columns(
        pl.col("_year_text")
        .cast(pl.Int32, strict=False)
        .alias("year")
    )
    .filter(pl.col("year").is_not_null())
    .group_by("year")
    .agg(pl.len().alias("records"))
    .sort("year")
    .collect()
)

if year_counts.height == 0:
    raise ValueError(
        "Tidak ada nilai pada kolom 'date' yang berhasil dikenali sebagai tahun."
    )

year_axis = pl.DataFrame(
    {"year": list(range(
        year_counts["year"].min(),
        year_counts["year"].max() + 1
    ))},
    schema={"year": pl.Int32},
)

yearly_trend = (
    year_axis
    .join(year_counts, on="year", how="left")
    .with_columns(pl.col("records").fill_null(0))
    .sort("year")
    .with_columns(
        pl.col("records")
        .rolling_mean(window_size=5)
        .alias("rolling_mean_5y")
    )
)

parsed_rows = year_counts["records"].sum()

print(f"Rentang tahun yang terdeteksi: "
      f"{year_counts['year'].min()}–{year_counts['year'].max()}")
print(f"Baris dengan tahun yang berhasil dikenali: {parsed_rows:,} dari {df.height:,}")
print(f"Baris tanpa tahun yang dikenali/tidak valid: {df.height - parsed_rows:,}")
print("\nTren tahunan dan rata-rata bergerak 5 tahun:")
print(yearly_trend.head(15))

Rentang tahun yang terdeteksi: -2900–2100
Baris dengan tahun yang berhasil dikenali: 2,046,934 dari 2,121,704
Baris tanpa tahun yang dikenali/tidak valid: 74,770

Tren tahunan dan rata-rata bergerak 5 tahun:
shape: (15, 3)
┌───────┬─────────┬─────────────────┐
│ year  ┆ records ┆ rolling_mean_5y │
│ ---   ┆ ---     ┆ ---             │
│ i32   ┆ u32     ┆ f64             │
╞═══════╪═════════╪═════════════════╡
│ -2900 ┆ 2       ┆ null            │
│ -2899 ┆ 0       ┆ null            │
│ -2898 ┆ 0       ┆ null            │
│ -2897 ┆ 0       ┆ null            │
│ -2896 ┆ 0       ┆ 0.4             │
│ …     ┆ …       ┆ …               │
│ -2890 ┆ 0       ┆ 0.0             │
│ -2889 ┆ 0       ┆ 0.0             │
│ -2888 ┆ 0       ┆ 0.0             │
│ -2887 ┆ 0       ┆ 0.0             │
│ -2886 ┆ 0       ┆ 0.0             │
└───────┴─────────┴─────────────────┘


## Bagian 3: Minimal 6 Visualisasi Interaktif (Plotly / Altair)
Bangun minimal 6 visualisasi interaktif yang mencakup:

- Visualisasi 1: Tren deret waktu (Line chart dengan range slider)
- Visualisasi 2: Distribusi variabel utama (Box plot / Histogram / Violin plot)
- Visualisasi 3: Korelasi antar-variabel numerik (Heatmap / Scatter plot sampling)
- Visualisasi 4: Komposisi kategori utama (Bar chart / Treemap)
- Visualisasi 5: Analisis siklus musiman / perbandingan kelompok
- Visualisasi 6: Pola spasial atau korelasi multi-dimensi

In [4]:
required_columns = {"text", "lang", "source", "dataset_split"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"Kolom berikut tidak ditemukan: {sorted(missing_columns)}")

if not {"year", "records", "rolling_mean_5y"}.issubset(yearly_trend.columns):
    raise ValueError(
        "yearly_trend belum tersedia. Jalankan kode Bagian 2 terlebih dahulu."
    )

df_eda = df.with_columns(
    pl.col("text").str.len_chars().alias("text_length")
)

In [5]:
# Visualisasi 1: Tren jumlah data berdasarkan tahun + rolling mean

fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=yearly_trend["year"].to_list(),
    y=yearly_trend["records"].to_list(),
    mode="lines",
    name="Jumlah data per tahun",
    line=dict(color="#3182bd")
))

fig1.add_trace(go.Scatter(
    x=yearly_trend["year"].to_list(),
    y=yearly_trend["rolling_mean_5y"].to_list(),
    mode="lines",
    name="Rata-rata bergerak 5 tahun",
    line=dict(color="#e6550d", width=3)
))

fig1.update_layout(
    title="Tren Data CommonLingua Berdasarkan Tahun pada Metadata",
    xaxis=dict(title="Tahun", rangeslider=dict(visible=True)),
    yaxis_title="Jumlah baris",
    hovermode="x unified"
)
fig1.show()

In [6]:
# Visualisasi 2: Distribusi panjang teks
# Histogram diringkas per 50 karakter; panjang 1000+ digabung

length_histogram = (
    df_eda
    .select(
        pl.when(pl.col("text_length") >= 1000)
        .then(pl.lit(1000))
        .otherwise((pl.col("text_length") // 50) * 50)
        .cast(pl.Int32)
        .alias("bin_start")
    )
    .group_by("bin_start")
    .agg(pl.len().alias("records"))
    .sort("bin_start")
)

histogram_labels = [
    "1000+" if start == 1000 else f"{start}-{start + 49}"
    for start in length_histogram["bin_start"].to_list()
]

fig2 = go.Figure(go.Bar(
    x=histogram_labels,
    y=length_histogram["records"].to_list(),
    marker_color="#31a354",
    hovertemplate="Panjang teks: %{x} karakter<br>Jumlah data: %{y:,}<extra></extra>"
))

fig2.update_layout(
    title="Distribusi Panjang Teks",
    xaxis_title="Rentang panjang teks (karakter)",
    yaxis_title="Jumlah baris"
)
fig2.show()

In [7]:
# Visualisasi 3: Hubungan jumlah data bahasa dengan rata-rata panjang teks

language_stats = (
    df_eda
    .group_by("lang")
    .agg(
        pl.len().alias("records"),
        pl.col("text_length").mean().alias("avg_text_length")
    )
)

fig3 = go.Figure(go.Scatter(
    x=language_stats["records"].to_list(),
    y=language_stats["avg_text_length"].to_list(),
    text=language_stats["lang"].to_list(),
    mode="markers",
    marker=dict(
        size=10,
        color=language_stats["avg_text_length"].to_list(),
        colorscale="Viridis",
        showscale=True,
        colorbar=dict(title="Rata-rata<br>karakter")
    ),
    hovertemplate=(
        "Bahasa: %{text}<br>"
        "Jumlah data: %{x:,}<br>"
        "Rata-rata panjang teks: %{y:.1f} karakter"
        "<extra></extra>"
    )
))

fig3.update_layout(
    title="Hubungan Jumlah Data dan Rata-rata Panjang Teks per Bahasa",
    xaxis=dict(title="Jumlah baris", type="log"),
    yaxis_title="Rata-rata panjang teks (karakter)"
)
fig3.show()

In [8]:
# Visualisasi 4: 20 bahasa dengan jumlah data terbanyak

top_languages = (
    df_eda
    .group_by("lang")
    .agg(pl.len().alias("records"))
    .sort("records", descending=True)
    .head(20)
)

fig4 = go.Figure(go.Bar(
    x=top_languages["lang"].to_list(),
    y=top_languages["records"].to_list(),
    marker_color="#756bb1",
    hovertemplate="Bahasa: %{x}<br>Jumlah data: %{y:,}<extra></extra>"
))

fig4.update_layout(
    title="20 Bahasa dengan Jumlah Data Terbanyak",
    xaxis_title="Kode bahasa",
    yaxis_title="Jumlah baris"
)
fig4.show()

In [9]:
# Visualisasi 5: Perbandingan bahasa teratas antara train dan validation

top_language_names = top_languages["lang"].to_list()
fig5 = go.Figure()

for split_name in ["train", "validation"]:
    split_counts = (
        df_eda
        .filter(
            (pl.col("dataset_split") == split_name)
            & pl.col("lang").is_in(top_language_names)
        )
        .group_by("lang")
        .agg(pl.len().alias("records"))
    )

    counts_by_language = dict(zip(
        split_counts["lang"].to_list(),
        split_counts["records"].to_list()
    ))

    fig5.add_trace(go.Bar(
        name=split_name,
        x=top_language_names,
        y=[counts_by_language.get(lang, 0) for lang in top_language_names]
    ))

fig5.update_layout(
    title="Perbandingan 20 Bahasa Teratas antara Train dan Validation",
    xaxis_title="Kode bahasa",
    yaxis_title="Jumlah baris",
    barmode="group"
)
fig5.show()

In [10]:
# Visualisasi 6: Bubble chart multi-dimensi per sumber
# Ukuran dan warna menunjukkan keragaman bahasa pada setiap sumber

source_stats = (
    df_eda
    .group_by("source")
    .agg(
        pl.len().alias("records"),
        pl.col("text_length").mean().alias("avg_text_length"),
        pl.col("lang").n_unique().alias("language_diversity")
    )
    .sort("records", descending=True)
    .head(25)
)

max_diversity = max(source_stats["language_diversity"].to_list())
bubble_sizes = [
    10 + 35 * diversity / max_diversity
    for diversity in source_stats["language_diversity"].to_list()
]

fig6 = go.Figure(go.Scatter(
    x=source_stats["records"].to_list(),
    y=source_stats["avg_text_length"].to_list(),
    text=source_stats["source"].to_list(),
    mode="markers",
    marker=dict(
        size=bubble_sizes,
        color=source_stats["language_diversity"].to_list(),
        colorscale="Plasma",
        showscale=True,
        colorbar=dict(title="Jumlah<br>bahasa"),
        opacity=0.8
    ),
    hovertemplate=(
        "Sumber: %{text}<br>"
        "Jumlah data: %{x:,}<br>"
        "Rata-rata panjang teks: %{y:.1f} karakter"
        "<extra></extra>"
    )
))

fig6.update_layout(
    title="Perbandingan Sumber: Jumlah Data, Panjang Teks, dan Keragaman Bahasa",
    xaxis=dict(title="Jumlah baris", type="log"),
    yaxis_title="Rata-rata panjang teks (karakter)"
)
fig6.show()

## Bagian 4: Rumusan 5 Insight Analitik Utama

1. **Insight 1 — Tren Waktu dan Musiman**  
   Kolom `date` berisi tahun dalam format teks, termasuk nilai seperti `-100` dan `-113`, serta memiliki nilai kosong sekitar 2,27% pada train dan 1,18% pada validation. Karena tanggal berasal dari metadata berbagai sumber, tren tahunan pada grafik menggambarkan sebaran tahun yang tercatat bukan pertumbuhan penggunaan bahasa dari waktu ke waktu. Data ini juga tidak cukup untuk menyimpulkan adanya pola musiman bulanan.

2. **Insight 2 — Distribusi dan Anomali**  
   Profiling Milestone 2 menunjukkan kolom `text` dan `lang` tidak memiliki nilai kosong, tetapi metadata `creator` kosong pada sekitar 99,93% data. Karena itu, `creator` kurang dapat diandalkan untuk segmentasi. Pada histogram, rentang teks `1000+` merupakan gabungan semua teks yang panjangnya minimal 1000 karakter; teks dalam kelompok ini perlu ditinjau sebelum dianggap sebagai kesalahan atau dihapus.

3. **Insight 3 — Hubungan Antarvariabel**  
   Scatter plot membandingkan jumlah baris dan rata-rata panjang teks untuk setiap bahasa. Grafik ini membantu memeriksa apakah bahasa dengan jumlah data lebih besar cenderung memiliki teks lebih panjang atau lebih pendek. Arah dan kekuatan hubungannya harus disimpulkan dari sebaran titik pada grafik; hubungan tersebut tidak membuktikan sebab-akibat.

4. **Insight 4 — Perilaku Kategori dan Pembagian Data**  
   Dataset mencakup sekitar 334 bahasa dengan total 2.755.443 baris: 2.482.568 baris train (sekitar 90,1%) dan 272.875 baris validation (sekitar 9,9%). Grafik perbandingan bahasa antar-split perlu diperiksa untuk memastikan bahasa-bahasa utama pada train juga terwakili pada validation; jumlah baris validation yang lebih kecil membuat pemeriksaan ini penting.

5. **Insight 5 — Rekomendasi Praktis**  
   Pertahankan teks panjang dan metadata sumber selama belum terbukti sebagai kesalahan, serta jangan menafsirkan kolom `date` sebagai deret waktu yang seragam. Untuk evaluasi yang lebih adil, pantau jumlah data per bahasa pada train dan validation, dan dokumentasikan keterbatasan metadata seperti banyaknya nilai `creator` yang kosong.